# Notebook 06b — Additional Experiments

Extends Notebook 06 with experiments not covered in the original run:

1. **Experiment 9** — Recovery-gap window sensitivity (6, 12, 24 month windows)
2. **Experiment 10** — Composite RIS weighting sensitivity (5 profiles, Jaccard + Spearman)
3. **Experiment 11** — GRU temporal baseline
4. **Experiment 12** — TempCNN (1D convolutional) temporal baseline

Prerequisites:
- Notebook 04 outputs: `X_scaled.npy`, `y_burned.npy`, `X_scaled_mask.npy`, `targets_aux.npz`, `splits.json`, `feature_names.json`
- Notebook 05 outputs: `model_main.pt`, `predictions_test.npz`

Outputs stored under `models/experiments/exp{9,10,11,12}_*/`.


In [ ]:
# --- Environment auto-detection ---
import sys, importlib, os

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"Environment: {'COLAB' if IS_COLAB else 'LOCAL'}")
print(f"Workspace: {THESIS_ROOT}")


In [ ]:
# --- Auto-download from Google Drive if running locally and files are missing ---
# On Colab this cell is a no-op (files are already on the mounted Drive).
# Locally, it checks which processed / model files are present under
# ~/thesisv4/ and downloads the missing ones via the Google Drive API.
#
# First-time setup (local, one-off): the browser will open once for
# authentication. After that, credentials are cached in
# ~/.config/gdrive_thesisv4/token.json.
#
# Total download size ~15-16 GB (X_scaled.npy is the largest at ~13 GB).
# Expected time: 30-90 minutes depending on connection speed.

REQUIRED_FILES = {
    # (path relative to THESIS_ROOT, approximate size in MB)
    "data/processed/X_scaled.npy":         13_000,
    "data/processed/X_scaled_mask.npy":     2_000,
    "data/processed/y_burned.npy":             65,
    "data/processed/targets_aux.npz":          10,
    "data/processed/splits.json":               1,
    "data/processed/feature_names.json":        1,
    "data/processed/scaler.pkl":                1,
    "models/main/model_main.pt":                1,
    "models/main/results_main.json":            1,
    "models/main/training_history.json":        1,
    "models/main/predictions_test.npz":       500,
}

def check_files():
    """Return list of files that need downloading."""
    missing = []
    for rel_path in REQUIRED_FILES:
        full_path = os.path.join(THESIS_ROOT, rel_path)
        if not os.path.exists(full_path):
            missing.append(rel_path)
    return missing

if IS_COLAB:
    print("Running on Colab — files already on mounted Drive. Skipping download.")
    missing = []
else:
    missing = check_files()
    if not missing:
        print(f"All {len(REQUIRED_FILES)} required files already present locally.")
    else:
        print(f"Missing {len(missing)} of {len(REQUIRED_FILES)} required files:")
        total_mb = sum(REQUIRED_FILES[f] for f in missing)
        for f in missing:
            print(f"  - {f}   (~{REQUIRED_FILES[f]} MB)")
        print(f"\nTotal to download: ~{total_mb/1024:.1f} GB")


In [ ]:
# --- Auto-download from Drive via rclone (local only, skipped on Colab) ---
# Prerequisites (one-time setup):
#   1. Install rclone:
#      - Ubuntu/Debian: sudo apt install rclone
#      - macOS:          brew install rclone
#      - Windows:        download from https://rclone.org/downloads/
#   2. Configure a Google Drive remote named 'gdrive':
#         rclone config    (choose 'n', name 'gdrive', type 'drive', scope 1)
#   3. Verify with:
#         rclone ls gdrive:thesisv4/data/processed | head
#
# If a different remote name is used, override RCLONE_REMOTE below.

import subprocess

RCLONE_REMOTE = "gdrive"     # rclone remote name for your Google Drive
DRIVE_ROOT    = "thesisv4"    # top-level folder under My Drive

if IS_COLAB:
    print("Running on Colab — files already on mounted Drive. Skipping download.")
elif not missing:
    print("All required files already present locally. Skipping download.")
else:
    # Check that rclone is installed and configured
    try:
        subprocess.run(["rclone", "version"],
                       check=True, capture_output=True, text=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        raise SystemExit(
            "rclone is not installed or not in PATH. "
            "Install it first: https://rclone.org/downloads/")

    check = subprocess.run(
        ["rclone", "listremotes"], capture_output=True, text=True)
    if f"{RCLONE_REMOTE}:" not in check.stdout:
        raise SystemExit(
            f"rclone remote '{RCLONE_REMOTE}' not configured. "
            f"Run 'rclone config' first.")

    print(f"Downloading {len(missing)} missing files via rclone...")
    print(f"Total size: ~{sum(REQUIRED_FILES[f] for f in missing) / 1024:.1f} GB")
    print("This can take 30-90 minutes depending on your internet speed.\n")

    for rel_path in missing:
        remote_path = f"{RCLONE_REMOTE}:{DRIVE_ROOT}/{rel_path}"
        local_path  = os.path.join(THESIS_ROOT, rel_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        print(f"Downloading {rel_path}  (~{REQUIRED_FILES[rel_path]} MB)")
        # rclone copyto: copies a single file source -> destination path
        result = subprocess.run(
            ["rclone", "copyto", remote_path, local_path,
             "--progress", "--transfers", "4", "--checkers", "4"],
            check=False,
        )
        if result.returncode != 0:
            raise RuntimeError(f"rclone failed for {rel_path}")
        size_mb = os.path.getsize(local_path) / 1024**2
        print(f"  Done — {size_mb:.1f} MB\n")

    print(f"All {len(missing)} files downloaded to {THESIS_ROOT}")

print("\nReady to load data.")


In [ ]:
import json, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

import config, model_defs, metrics
importlib.reload(config); importlib.reload(model_defs); importlib.reload(metrics)

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
EXP_ROOT = config.PATHS["models_exp"]
os.makedirs(EXP_ROOT, exist_ok=True)
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# --- Load all shared data ---
X       = np.load(config.FILES["X_scaled"])
y       = np.load(config.FILES["y_burned"])
mask    = np.load(config.FILES["X_scaled"].replace(".npy", "_mask.npy"))
targets = dict(np.load(config.FILES["targets_aux"]))

with open(config.FILES["splits"]) as f:
    splits = json.load(f)
with open(config.FILES["feature_names"]) as f:
    feature_names = json.load(f)

idx_train = np.array(splits["train"])
idx_val   = np.array(splits["val"])
idx_test  = np.array(splits["test"])

N, T, C = X.shape
print(f"X shape:  {X.shape}")
print(f"Train:    {len(idx_train):,}")
print(f"Val:      {len(idx_val):,}")
print(f"Test:     {len(idx_test):,}")
print(f"Features: {C}")


---
## Experiment 9 — Recovery-gap window sensitivity

Recomputes the `recovery_gap` target using three window configurations:
- 6-month windows (Jan-Jun 2017 vs Jul-Dec 2025)
- 12-month windows (main setup: Jan-Dec 2017 vs Jan-Dec 2025)
- 24-month windows (2017-2018 vs 2024-2025)

Uses the raw (non-standardized) NBR values reconstructed from `X_scaled` via the saved scaler.


In [ ]:
# Reconstruct raw NBR from X_scaled via the saved scaler
with open(config.FILES["scaler"], "rb") as f:
    scaler_bundle = pickle.load(f)

scaler = scaler_bundle["scaler"]
spectral_cols = scaler_bundle["spectral_cols"]

# spectral_cols holds the *column indices into X* that were standardised, not
# feature names -- the scaler was fitted on X[..., spectral_cols], so NBR's
# statistics live at its position within that list, not at its index in X.
nbr_col_idx_in_X = feature_names.index("NBR")
nbr_pos_in_scaler = spectral_cols.index(nbr_col_idx_in_X)
nbr_mean = scaler.mean_[nbr_pos_in_scaler]
nbr_std  = scaler.scale_[nbr_pos_in_scaler]
print(f"NBR: column {nbr_col_idx_in_X} of X, position {nbr_pos_in_scaler} in the scaler")
print(f"     mean={nbr_mean:.4f}  scale={nbr_std:.4f}")

nbr_raw = X[:, :, nbr_col_idx_in_X] * nbr_std + nbr_mean
print(f"Reconstructed nbr_raw shape: {nbr_raw.shape}")
print(f"nbr_raw range: [{nbr_raw.min():.3f}, {nbr_raw.max():.3f}]  mean={nbr_raw.mean():.3f}")

In [ ]:
def compute_recovery_gap(nbr_raw, y, baseline_months, recent_months):
    """Compute recovery_gap with configurable window sizes.

    Follows the v4 convention: recovery_gap = 0 for pixels with no fire
    history; otherwise computed as normalized difference between mean NBR
    in the first `baseline_months` and the last `recent_months`.

    Vectorised. The original per-pixel Python loop ran 608,922 iterations
    per configuration (1.83 M across the three configurations below), each
    allocating its own np.where result and two nanmean temporaries. The
    array form is numerically identical -- the nanmeans accumulate in
    float64 exactly as the per-pixel float() casts did -- and its
    temporaries are one NBR window wide (N x window), never the full cube.
    """
    burned = y.sum(axis=1) > 0    # pixels with at least one fire month

    baseline_nbr = np.nanmean(
        nbr_raw[:, :baseline_months].astype(np.float64), axis=1)
    recent_nbr = np.nanmean(
        nbr_raw[:, -recent_months:].astype(np.float64), axis=1)

    gap = (baseline_nbr - recent_nbr) / (np.abs(baseline_nbr) + 1e-6)
    gap = np.clip(gap, 0, 1)

    recovery_gap = np.zeros(len(y), dtype=np.float32)
    recovery_gap[burned] = gap[burned].astype(np.float32)
    return recovery_gap


configs = [
    ("6-month",  6,  6),
    ("12-month (main)", 12, 12),
    ("24-month", 24, 24),
]

rg_results = {}
for label, base_m, rec_m in configs:
    print(f"Computing recovery_gap for {label} windows...")
    rg = compute_recovery_gap(nbr_raw, y, base_m, rec_m)
    rg_results[label] = rg
    print(f"  Mean: {rg.mean():.4f}   Std: {rg.std():.4f}   Non-zero: {(rg > 0).sum():,}")

# Correlate alternative windows against the main (12-month) window on burned pixels
main_rg = rg_results["12-month (main)"]
burned_mask = main_rg > 0

print("\nCorrelations vs main (12-month) target, burned pixels only:")
comparison_rows = []
for label, rg in rg_results.items():
    if label == "12-month (main)":
        comparison_rows.append({
            "config": label, "vs_main_pearson": 1.0,
            "vs_main_spearman": 1.0,
            "mean_burned": rg[burned_mask].mean(),
            "std_burned":  rg[burned_mask].std(),
        })
        continue
    r, _ = pearsonr(rg[burned_mask], main_rg[burned_mask])
    s, _ = spearmanr(rg[burned_mask], main_rg[burned_mask])
    print(f"  {label:20s}  Pearson={r:.4f}  Spearman={s:.4f}")
    comparison_rows.append({
        "config": label,
        "vs_main_pearson": r,
        "vs_main_spearman": s,
        "mean_burned": rg[burned_mask].mean(),
        "std_burned":  rg[burned_mask].std(),
    })

df_rg = pd.DataFrame(comparison_rows)

EXP9_DIR = os.path.join(EXP_ROOT, "exp9_recovery_gap_windows")
os.makedirs(EXP9_DIR, exist_ok=True)
df_rg.to_csv(os.path.join(EXP9_DIR, "results.csv"), index=False)
with open(os.path.join(EXP9_DIR, "results.json"), "w") as f:
    json.dump({"windows": comparison_rows}, f, indent=2, default=float)
print(f"\nSaved: {EXP9_DIR}")

---
## Experiment 10 — Composite RIS weighting sensitivity

Compares five composite RIS profiles applied to the main model's test-set
predictions:
- equal (0.25 / 0.25 / 0.25 / 0.25)
- severity-dominant (0.50 / 0.20 / 0.20 / 0.10)
- persistence-dominant (0.20 / 0.50 / 0.20 / 0.10)
- recovery-gap-dominant (0.20 / 0.20 / 0.50 / 0.10)
- recurrence-dominant (0.20 / 0.20 / 0.10 / 0.50)

Order: severity / persistence / recovery_gap / recurrence.

Metrics: pairwise Jaccard overlap of top-10% pixels, pairwise Spearman rank correlation.


In [ ]:
# Load test-set predictions from Notebook 05
preds_path = os.path.join(config.PATHS["models_main"], "predictions_test.npz")
if not os.path.exists(preds_path):
    raise FileNotFoundError(
        f"Predictions file not found: {preds_path}. "
        "Re-run Notebook 05 with predictions save enabled.")

preds = np.load(preds_path)
print(f"Prediction keys: {list(preds.files)}")

sev  = preds["severity"]
per_ = preds["persistence"]
rec_ = preds["recovery_gap"]
rcr  = preds["recurrence"] if "recurrence" in preds.files else targets["recurrence"][idx_test]

n = len(sev)
print(f"Test-set predictions: {n:,}")


In [ ]:
profiles = {
    "equal":                 (0.25, 0.25, 0.25, 0.25),
    "severity_dominant":     (0.50, 0.20, 0.20, 0.10),
    "persistence_dominant":  (0.20, 0.50, 0.20, 0.10),
    "recovery_gap_dominant": (0.20, 0.20, 0.50, 0.10),
    "recurrence_dominant":   (0.20, 0.20, 0.10, 0.50),
}

ris_scores = {}
for name, (ws, wp, wrg, wrc) in profiles.items():
    ris = ws * sev + wp * per_ + wrg * rec_ + wrc * rcr
    ris_scores[name] = ris
    print(f"{name:25s}  mean={ris.mean():.4f}  std={ris.std():.4f}")

# --- Pairwise Jaccard overlap of top-10% pixels ---
def top_k_set(scores, k_frac=0.10):
    k = int(len(scores) * k_frac)
    return set(np.argsort(scores)[-k:])

top_sets = {name: top_k_set(s) for name, s in ris_scores.items()}

names = list(profiles.keys())
jaccard_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        inter = len(top_sets[a] & top_sets[b])
        union = len(top_sets[a] | top_sets[b])
        jaccard_matrix.loc[a, b] = inter / union if union > 0 else 0.0

print("\nJaccard overlap of top-10% pixels:")
print(jaccard_matrix.round(3).to_string())

# --- Pairwise Spearman rank correlation ---
spearman_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        rho, _ = spearmanr(ris_scores[a], ris_scores[b])
        spearman_matrix.loc[a, b] = rho

print("\nSpearman rank correlation of full rankings:")
print(spearman_matrix.round(3).to_string())

EXP10_DIR = os.path.join(EXP_ROOT, "exp10_composite_weighting")
os.makedirs(EXP10_DIR, exist_ok=True)
jaccard_matrix.to_csv(os.path.join(EXP10_DIR, "jaccard.csv"))
spearman_matrix.to_csv(os.path.join(EXP10_DIR, "spearman.csv"))
with open(os.path.join(EXP10_DIR, "results.json"), "w") as f:
    json.dump({
        "profiles": {k: list(v) for k, v in profiles.items()},
        "jaccard":  jaccard_matrix.to_dict(),
        "spearman": spearman_matrix.to_dict(),
    }, f, indent=2, default=float)
print(f"\nSaved: {EXP10_DIR}")


---
## Experiment 11 — GRU temporal baseline

A gated recurrent unit encoder trained on the same features, targets,
and splits as L-TAE. Provides a recurrent-architecture point of
comparison. All hyperparameters aligned with L-TAE where possible
(hidden dim = 128 = `d_model`; dropout = 0.1).


In [ ]:
class GRUModel(nn.Module):
    """Bidirectional GRU encoder with multi-task heads matching L-TAE."""
    def __init__(self, n_features, hidden_dim=128, n_layers=1, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=True,
        )
        d_out = hidden_dim * 2  # bidirectional
        self.dropout = nn.Dropout(dropout)

        # Per-time-step head for burned classification
        self.burn_head = nn.Linear(d_out, 1)

        # Sequence-summary heads for regression
        self.severity_head     = nn.Linear(d_out, 1)
        self.persistence_head  = nn.Linear(d_out, 1)
        self.recovery_gap_head = nn.Linear(d_out, 1)

    def forward(self, x, mask=None):
        # x: (B, T, C), mask: (B, T)
        out, _ = self.gru(x)               # (B, T, 2H)
        out = self.dropout(out)

        # Burned per month
        burn_logits = self.burn_head(out).squeeze(-1)  # (B, T)

        # Sequence pooling for regression — mean over valid time steps
        if mask is not None:
            m = mask.unsqueeze(-1).float()             # (B, T, 1)
            seq_mean = (out * m).sum(dim=1) / (m.sum(dim=1) + 1e-6)
        else:
            seq_mean = out.mean(dim=1)

        return {
            "burned_logits": burn_logits,
            "severity":     torch.sigmoid(self.severity_head(seq_mean)).squeeze(-1),
            "persistence":  torch.sigmoid(self.persistence_head(seq_mean)).squeeze(-1),
            "recovery_gap": torch.sigmoid(self.recovery_gap_head(seq_mean)).squeeze(-1),
        }


def train_and_evaluate_baseline(model, model_name, exp_dir, epochs=None):
    """Train a baseline model on the shared splits and evaluate it on test.

    Loaders come from metrics.make_loader, which gathers each batch out of
    the parent arrays on demand. The earlier local make_loader built
    TensorDataset(torch.from_numpy(X[indices]), ...) instead, which
    materialised a full copy of each split: 9.6 GB for train, 2.1 GB each
    for val and test, so all three together duplicated the entire 13.7 GB
    cube on top of the resident X -- the same pattern that OOM-killed the
    kernel in Notebook 06 (see the note on make_window_loader there). The
    two forked num_workers made it worse by touching those pages again.
    Batch composition is unchanged.

    Training, threshold tuning and test evaluation all go through the
    metrics helpers, so these baselines are scored on exactly the same code
    path as the L-TAE runs in Notebook 06 and are directly comparable.
    """
    if epochs is None:
        epochs = config.TRAINING["epochs"]

    os.makedirs(exp_dir, exist_ok=True)
    bs = config.TRAINING["batch_size"]
    train_loader = metrics.make_loader(X, y, mask, targets, idx_train, bs, True)
    val_loader   = metrics.make_loader(X, y, mask, targets, idx_val,   bs, False)
    test_loader  = metrics.make_loader(X, y, mask, targets, idx_test,  bs, False)

    criterion = model_defs.MaskedMultiTaskLoss(
        alpha=config.LOSS["focal_alpha"],
        gamma=config.LOSS["focal_gamma"],
        lambda_aux=config.LOSS["lambda_aux"],
    )
    optimizer = optim.AdamW(model.parameters(),
                            lr=config.TRAINING["lr"],
                            weight_decay=config.TRAINING["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_auc = -1.0
    best_state = None
    history = {"train": [], "val": []}

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        # compute_auc=False on the training pass: the AUC over every training
        # timestep costs more than the training step itself and is only ever
        # written to history.json. Model selection uses the validation AUC.
        train_m = metrics.run_epoch(model, criterion, optimizer, train_loader,
                                    DEVICE, train=True,
                                    grad_clip=config.TRAINING["grad_clip"],
                                    compute_auc=False)
        val_m = metrics.run_epoch(model, criterion, optimizer, val_loader,
                                  DEVICE, train=False)
        scheduler.step()
        history["train"].append(train_m)
        history["val"].append(val_m)

        if val_m["auc"] > best_val_auc:
            best_val_auc = float(val_m["auc"])
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        done = time.time() - t0
        eta = done / epoch * (epochs - epoch)
        print(f"{model_name}, epoch {epoch}/{epochs} - "
              f"train_loss {train_m['total']:.4f}, val_loss {val_m['total']:.4f}, "
              f"val_auc {val_m['auc']:.4f}"
              f"  [elapsed {done/60:.1f}m, eta {eta/60:.1f}m]", flush=True)

    # --- Load best model, tune threshold on val, evaluate on test ---
    model.load_state_dict(best_state)
    best_thr, best_f1, _, _, _ = metrics.tune_threshold(model, val_loader, DEVICE)
    burned_metrics, correlations, predictions = metrics.evaluate_test_set(
        model, test_loader, DEVICE, best_thr, targets, idx_test)

    n_params = sum(p.numel() for p in model.parameters())
    extra = {
        "model_info": {
            "architecture":   model.__class__.__name__,
            "n_parameters":   int(n_params),
            "epochs_trained": epochs,
            "best_val_auc":   float(best_val_auc),
            "loss_type":      "MaskedMultiTaskLoss",
        },
        "data_info": {"n_train": len(idx_train),
                      "n_val":   len(idx_val),
                      "n_test":  len(idx_test)},
    }
    metrics.save_experiment_results(
        exp_dir, model_name, best_state, history,
        burned_metrics, correlations, per_category=None, extra=extra)

    print(f"\n{model_name} test results:")
    print(f"  N parameters: {n_params:,}")
    print(f"  AUC: {burned_metrics['auc']:.4f}  F1: {burned_metrics['f1']:.4f}  "
          f"(threshold {best_thr:.3f})")
    print(f"  Precision: {burned_metrics['precision']:.4f}  "
          f"Recall: {burned_metrics['recall']:.4f}")
    for dim, c in correlations.items():
        print(f"  {dim:15s}  Pearson={c['pearson']:.4f}  Spearman={c['spearman']:.4f}")

    return {
        "experiment_name":  model_name,
        "model_info":       extra["model_info"],
        "burned_detection": burned_metrics,
        "correlations":     correlations,
    }


# --- Train GRU ---
config.set_seed()
gru = GRUModel(n_features=C, hidden_dim=128, n_layers=1, dropout=0.1).to(DEVICE)
print(f"GRU parameters: {sum(p.numel() for p in gru.parameters()):,}")

EXP11_DIR = os.path.join(EXP_ROOT, "exp11_gru_baseline")
gru_results = train_and_evaluate_baseline(gru, "exp11_gru_baseline", EXP11_DIR)

---
## Experiment 12 — TempCNN temporal baseline

1D convolutional temporal encoder inspired by Pelletier et al. (2019),
adapted for the multi-task setup. Provides a convolutional-architecture
point of comparison against L-TAE (attention-based) and GRU (recurrent).


In [ ]:
class TempCNNBlock(nn.Module):
    """Conv1d + BatchNorm + ReLU + Dropout block for temporal features."""
    def __init__(self, in_channels, out_channels, kernel_size=5, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv1d(in_channels, out_channels,
                              kernel_size=kernel_size, padding=pad)
        self.bn   = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.relu(self.bn(self.conv(x))))


class TempCNNModel(nn.Module):
    """Three-block TempCNN encoder with multi-task heads matching L-TAE.

    Kernel size 5 in each block; dilation kept at 1. Followed by a linear
    layer over the flattened temporal features for the regression heads,
    and per-time-step logits for classification.
    """
    def __init__(self, n_features, hidden_dim=128, dropout=0.1, kernel_size=5):
        super().__init__()
        self.block1 = TempCNNBlock(n_features, hidden_dim, kernel_size, dropout)
        self.block2 = TempCNNBlock(hidden_dim, hidden_dim, kernel_size, dropout)
        self.block3 = TempCNNBlock(hidden_dim, hidden_dim, kernel_size, dropout)

        # Per-time-step burned classification
        self.burn_head = nn.Conv1d(hidden_dim, 1, kernel_size=1)

        # Global pooled regression heads
        self.severity_head     = nn.Linear(hidden_dim, 1)
        self.persistence_head  = nn.Linear(hidden_dim, 1)
        self.recovery_gap_head = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask=None):
        # x: (B, T, C) -> (B, C, T) for Conv1d
        h = x.transpose(1, 2)
        h = self.block1(h)
        h = self.block2(h)
        h = self.block3(h)          # (B, hidden, T)

        burn_logits = self.burn_head(h).squeeze(1)  # (B, T)

        # Masked global average pool for regression
        if mask is not None:
            m = mask.unsqueeze(1).float()   # (B, 1, T)
            pooled = (h * m).sum(dim=2) / (m.sum(dim=2) + 1e-6)
        else:
            pooled = h.mean(dim=2)          # (B, hidden)

        return {
            "burned_logits": burn_logits,
            "severity":     torch.sigmoid(self.severity_head(pooled)).squeeze(-1),
            "persistence":  torch.sigmoid(self.persistence_head(pooled)).squeeze(-1),
            "recovery_gap": torch.sigmoid(self.recovery_gap_head(pooled)).squeeze(-1),
        }


# cuDNN convolutions are broken in this environment: the wheel pair here
# (torch 2.13.0+cu130 with nvidia-cudnn-cu13 9.20.0.48) raises
# CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH on every Conv1d forward. Turning
# cuDNN off routes convolutions through PyTorch's native CUDA kernels, which
# produce the same result. The GRU above is left on the cuDNN path, where it
# works and is much faster. Remove this once the cuDNN wheel is realigned.
if DEVICE == "cuda":
    torch.backends.cudnn.enabled = False
    print("cuDNN disabled for TempCNN (sublibrary version mismatch in this env)")

# --- Train TempCNN ---
config.set_seed()
tempcnn = TempCNNModel(n_features=C, hidden_dim=128, dropout=0.1, kernel_size=5).to(DEVICE)
print(f"TempCNN parameters: {sum(p.numel() for p in tempcnn.parameters()):,}")

EXP12_DIR = os.path.join(EXP_ROOT, "exp12_tempcnn_baseline")
tempcnn_results = train_and_evaluate_baseline(tempcnn, "exp12_tempcnn_baseline", EXP12_DIR)

---
## Summary table

Aggregate the new experiments alongside the main model and existing
Random Forest baselines for a single comparison table.


In [ ]:
# Load main model results and existing baselines
summary_rows = []

with open(os.path.join(config.PATHS["models_main"], "results_main.json")) as f:
    main_res = json.load(f)
summary_rows.append({
    "model": "L-TAE (main)",
    "n_params": main_res["model_info"]["n_parameters"],
    "auc": main_res["burned_detection"]["auc"],
    "f1":  main_res["burned_detection"]["f1"],
    "severity_pearson":     main_res["correlations"]["severity"]["pearson"],
    "persistence_pearson":  main_res["correlations"]["persistence"]["pearson"],
    "recovery_gap_pearson": main_res["correlations"]["recovery_gap"]["pearson"],
})

# RF baselines from exp5
try:
    with open(os.path.join(EXP_ROOT, "exp5_baselines", "results.json")) as f:
        rf_res = json.load(f)
    summary_rows.append({
        "model": "RF (raw)",
        "n_params": rf_res["rf_raw"].get("n_features", "—"),
        "auc": None, "f1": None,
        "severity_pearson":     rf_res["rf_raw"]["correlations"]["severity"]["pearson"],
        "persistence_pearson":  rf_res["rf_raw"]["correlations"]["persistence"]["pearson"],
        "recovery_gap_pearson": rf_res["rf_raw"]["correlations"]["recovery_gap"]["pearson"],
    })
    summary_rows.append({
        "model": "RF (PCA)",
        "n_params": rf_res["rf_pca"].get("n_components", "—"),
        "auc": None, "f1": None,
        "severity_pearson":     rf_res["rf_pca"]["correlations"]["severity"]["pearson"],
        "persistence_pearson":  rf_res["rf_pca"]["correlations"]["persistence"]["pearson"],
        "recovery_gap_pearson": rf_res["rf_pca"]["correlations"]["recovery_gap"]["pearson"],
    })
except FileNotFoundError:
    print("Warning: exp5_baselines results not found")

# GRU + TempCNN
for label, res in [("GRU", gru_results), ("TempCNN", tempcnn_results)]:
    summary_rows.append({
        "model": label,
        "n_params": res["model_info"]["n_parameters"],
        "auc": res["burned_detection"]["auc"],
        "f1":  res["burned_detection"]["f1"],
        "severity_pearson":     res["correlations"]["severity"]["pearson"],
        "persistence_pearson":  res["correlations"]["persistence"]["pearson"],
        "recovery_gap_pearson": res["correlations"]["recovery_gap"]["pearson"],
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 90)
print("BASELINE COMPARISON SUMMARY")
print("=" * 90)
print(summary_df.to_string(index=False))

summary_out = os.path.join(config.PATHS["outputs_tab"], "baseline_comparison.csv")
os.makedirs(os.path.dirname(summary_out), exist_ok=True)
summary_df.to_csv(summary_out, index=False)
print(f"\nSaved: {summary_out}")
